In [1]:
import os
import sys
from google.colab import drive

# 1.1 Mount Drive and set Project Path
drive.mount('/content/drive', force_remount=True)
PROJECT_PATH = '/content/drive/MyDrive/CILP_Project'
sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# 1.2 Unzip Data to local runtime
if not os.path.exists('/content/data/assessment'):
    print(" Extracting multimodal data...")
    !unzip -q assessment.zip -d /content/data/
    print(" Extraction complete.")

Mounted at /content/drive
 Extracting multimodal data...
 Extraction complete.


In [2]:
!pip install -r "{PROJECT_PATH}/requirements_colab.txt"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.3/316.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.5/934.5 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.3 MB/s eta 

In [3]:
import os
import sys
import random
import numpy as np
import torch
import wandb
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from src.training import train_one_epoch
import torch.nn as nn
import torch.nn.functional as F
import importlib
import src.models
importlib.reload(src.models)

from src.models import CILP_Model, Projector, RGB2LiDARClassifier, Encoder
from torch.optim.lr_scheduler import CosineAnnealingLR

# --- REPRODUCIBILITY---
def set_seeds(seed=51):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

# --- DATASET CLASS ---
class MultimodalDataset(Dataset):
    def __init__(self, root_dir, subset=1.0):
        self.samples = []
        classes = {'cubes': 0, 'spheres': 1}
        print(f"Scanning {root_dir}...")

        for cls_name, label in classes.items():
            rgb_dir = os.path.join(root_dir, cls_name, 'rgb')
            lidar_dir = os.path.join(root_dir, cls_name, 'lidar')
            if not os.path.exists(rgb_dir): continue

            rgb_f = sorted([f for f in os.listdir(rgb_dir) if f.endswith('.png')])
            lidar_f = sorted([f for f in os.listdir(lidar_dir) if f.endswith('.npy')])

            # Truncate to match lengths
            limit = min(len(rgb_f), len(lidar_f))
            for r, l in zip(rgb_f[:limit], lidar_f[:limit]):
                self.samples.append({
                    'rgb': os.path.join(rgb_dir, r),
                    'lidar': os.path.join(lidar_dir, l),
                    'label': label
                })

        # CRITICAL FIX: Shuffle for Class Balance
        random.seed(42)
        random.shuffle(self.samples)

        if subset < 1.0:
            self.samples = self.samples[:int(len(self.samples)*subset)]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        rgb = transforms.ToTensor()(Image.open(item['rgb']).convert("RGBA"))
        lidar = torch.from_numpy(np.load(item['lidar'])).float().unsqueeze(0)
        return rgb, lidar, torch.tensor(item['label'], dtype=torch.float)

# --- INITIALIZATION ---
set_seeds(51)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = '/content/data/assessment'
CHECKPOINT_DIR = '/content/drive/MyDrive/CILP_Project/checkpoints'
if not os.path.exists(CHECKPOINT_DIR): os.makedirs(CHECKPOINT_DIR)

# LOADERS
print("\n--- Initializing DataLoaders ---")
train_loader = DataLoader(MultimodalDataset(DATA_ROOT, subset=0.2), batch_size=32, shuffle=True)
val_loader = DataLoader(MultimodalDataset(DATA_ROOT, subset=0.04), batch_size=32)
print(" Data Ready.")


--- Initializing DataLoaders ---
Scanning /content/data/assessment...
Scanning /content/data/assessment...
 Data Ready.


In [4]:
import torch.nn as nn
import torch.nn.functional as F

# Best Downsampling (Strided Conv) ---
class EmbedderStrided(nn.Module):
    def __init__(self, in_channels, emb_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 50, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(50, 50, kernel_size=3, stride=2, padding=1), nn.ReLU(), # Strided
            nn.Conv2d(50, 100, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(100, 100, kernel_size=3, stride=2, padding=1), nn.ReLU(), # Strided
            nn.Conv2d(100, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.head = nn.Linear(128, emb_size)
    def forward(self, x): return self.head(self.features(x))

class CILP_Model(nn.Module):
    def __init__(self, emb_size=128):
        super().__init__()
        # Enforcing Strided Encoders
        self.rgb_encoder = EmbedderStrided(4, emb_size)
        self.lidar_encoder = EmbedderStrided(1, emb_size)

    def forward(self, rgb, lidar):
        r_emb = F.normalize(self.rgb_encoder(rgb), p=2, dim=1)
        l_emb = F.normalize(self.lidar_encoder(lidar), p=2, dim=1)
        return r_emb, l_emb

class Projector(nn.Module):
    def __init__(self, input_dim=128, output_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Linear(256, output_dim))
    def forward(self, x): return self.net(x)

class FinalClassifier(nn.Module):
    def __init__(self, input_dim=128):
        super().__init__()
        self.head = nn.Linear(input_dim, 1)
    def forward(self, x): return self.head(x)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
import os
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
from src.training import train_one_epoch

# --- CONFIGURATION ---
CONFIG = {
    "project_name": "cilp-extended-assessment",
    "architecture": "CILP_MaxPool",
    "batch_size": 32,
    "embedding_size": 200,
    "lr": 1e-3,
    "epochs": {"stage1": 50, "stage2": 50, "stage3": 50},
    "downsampling": "MaxPool"
}

if wandb.run is not None: wandb.finish()
wandb.init(project=CONFIG["project_name"], config=CONFIG, reinit=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- MODEL DEFINITIONS ---
class Encoder(nn.Module):
    def __init__(self, in_channels, emb_size=200):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(100, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.head = nn.Linear(128, emb_size)
    def forward(self, x): return self.head(self.features(x))

class CILP_Model(nn.Module):
    def __init__(self, emb_size=200):
        super().__init__()
        self.rgb_encoder = Encoder(4, emb_size)
        self.lidar_encoder = Encoder(1, emb_size)
    def forward(self, rgb, lidar):
        r = F.normalize(self.rgb_encoder(rgb), p=2, dim=1)
        l = F.normalize(self.lidar_encoder(lidar), p=2, dim=1)
        return r, l

class Projector(nn.Module):
    def __init__(self, input_dim=200, output_dim=200):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Linear(256, output_dim))
    def forward(self, x): return self.net(x)

class RGB2LiDARClassifier(nn.Module):
    def __init__(self, rgb_encoder, projector, input_dim=200):
        super().__init__()
        self.rgb_encoder = rgb_encoder
        self.projector = projector
        for p in self.rgb_encoder.parameters(): p.requires_grad = False
        for p in self.projector.parameters(): p.requires_grad = False
        self.classifier = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, rgb):
        with torch.no_grad(): emb = self.projector(self.rgb_encoder(rgb))
        return self.classifier(emb)

# --- STAGE 1: CONTRASTIVE PRETRAINING ---
print("Starting Stage 1")
cilp = CILP_Model(CONFIG["embedding_size"]).to(device)
opt = torch.optim.Adam(cilp.parameters(), lr=CONFIG["lr"])
sched = CosineAnnealingLR(opt, T_max=CONFIG["epochs"]["stage1"], eta_min=1e-6)

wandb.config.update({"cilp_params": sum(p.numel() for p in cilp.parameters())})

for epoch in range(CONFIG["epochs"]["stage1"]):
    train_loss = train_one_epoch(cilp, train_loader, opt, device, is_cilp=True)
    sched.step()

    cilp.eval()
    val_loss = 0.0
    all_r, all_l = [], []
    with torch.no_grad():
        for rgb, lidar, _ in val_loader:
            r, l = cilp(rgb.to(device), lidar.to(device))
            logits = torch.matmul(r, l.T) / 0.07
            targets = torch.arange(r.size(0)).to(device)
            val_loss += (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)).item() / 2
            if len(all_r) < 1: all_r.append(r); all_l.append(l)
    val_loss /= len(val_loader)
    wandb.log({"stage": 1, "val_loss": val_loss, "epoch": epoch})
    if (epoch+1) % 10 == 0: print(f"Ep {epoch+1}: Val Loss {val_loss:.4f}")

sim_matrix = torch.matmul(all_r[0], all_l[0].T).cpu().numpy()
plt.figure(figsize=(6,5))
plt.imshow(sim_matrix, cmap='viridis')
wandb.log({"similarity_matrix": wandb.Image(plt)})
plt.close()
torch.save(cilp.state_dict(), os.path.join(CHECKPOINT_DIR, "CILP_Stage1.pth"))

# --- STAGE 2: PROJECTOR ---
print("Starting Stage 2")
proj = Projector(CONFIG["embedding_size"], CONFIG["embedding_size"]).to(device)
opt_proj = torch.optim.Adam(proj.parameters(), lr=CONFIG["lr"])
sched_proj = CosineAnnealingLR(opt_proj, T_max=CONFIG["epochs"]["stage2"], eta_min=1e-6)
mse = nn.MSELoss()

for epoch in range(CONFIG["epochs"]["stage2"]):
    proj.train()
    for rgb, lidar, _ in train_loader:
        opt_proj.zero_grad()
        with torch.no_grad(): r, l = cilp(rgb.to(device), lidar.to(device))
        loss = mse(proj(r), l)
        loss.backward()
        opt_proj.step()
    sched_proj.step()

    proj.eval()
    val_mse = 0
    with torch.no_grad():
        for rgb, lidar, _ in val_loader:
            r, l = cilp(rgb.to(device), lidar.to(device))
            val_mse += mse(proj(r), l).item()
    val_mse /= len(val_loader)
    wandb.log({"stage": 2, "val_mse": val_mse, "epoch": epoch})
    if (epoch+1) % 10 == 0: print(f"Ep {epoch+1}: Val MSE {val_mse:.4f}")

torch.save(proj.state_dict(), os.path.join(CHECKPOINT_DIR, "CILP_Projector.pth"))

# --- STAGE 3: CLASSIFIER ---
print("Starting Stage 3")
final_model = RGB2LiDARClassifier(cilp.rgb_encoder, proj, CONFIG["embedding_size"]).to(device)
opt_cls = torch.optim.Adam(final_model.classifier.parameters(), lr=CONFIG["lr"])
sched_cls = CosineAnnealingLR(opt_cls, T_max=CONFIG["epochs"]["stage3"], eta_min=1e-6)

for epoch in range(CONFIG["epochs"]["stage3"]):
    final_model.classifier.train()
    for rgb, lidar, labels in train_loader:
        opt_cls.zero_grad()
        out = final_model(rgb.to(device)).view(-1)
        loss = F.binary_cross_entropy_with_logits(out, labels.to(device).float())
        loss.backward()
        opt_cls.step()
    sched_cls.step()

    final_model.eval()
    correct, total = 0, 0  # <--- FIXED LINE HERE
    sample_imgs = []
    with torch.no_grad():
        for i, (rgb, lidar, labels) in enumerate(val_loader):
            out = final_model(rgb.to(device)).view(-1)
            preds = (torch.sigmoid(out) > 0.5).float()
            correct += (preds == labels.to(device)).sum().item()
            total += labels.size(0)
            if epoch == CONFIG["epochs"]["stage3"]-1 and len(sample_imgs) < 5:
                for j in range(min(5, len(rgb))):
                    img = rgb[j].permute(1,2,0).cpu().numpy()
                    sample_imgs.append(wandb.Image(img, caption=f"P:{int(preds[j])} T:{int(labels[j])}"))

    val_acc = correct / total
    wandb.log({"stage": 3, "val_acc": val_acc, "epoch": epoch, "predictions": sample_imgs})
    if (epoch+1) % 10 == 0: print(f"Ep {epoch+1}: Val Acc {val_acc:.4f}")

wandb.finish()
print("Task 5 Done")

epoch,▁▁▂▂▂▃▄▄▄▅▅▆▆▆▆▇█▁▁▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▇▇█
stage,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█████████████████████
val_loss,███████▇██▇▇▆▇▆▆▆▆▆▅▄▄▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_mse,█▇▅▅▅▅▅▅▅▇▅▅▃▄▄▄▅▃▂▄▂▂▂▃▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁
epoch,49
stage,2
val_loss,2.17145
val_mse,0.00061


Starting Stage 1


Ep 10: Val Loss 3.0708


Ep 20: Val Loss 2.3381


Ep 30: Val Loss 1.8891


Ep 40: Val Loss 1.5536


Ep 50: Val Loss 1.4969
Starting Stage 2
Ep 10: Val MSE 0.0008
Ep 20: Val MSE 0.0008
Ep 30: Val MSE 0.0008
Ep 40: Val MSE 0.0007
Ep 50: Val MSE 0.0007
Starting Stage 3
Ep 10: Val Acc 0.9302
Ep 20: Val Acc 0.9302
Ep 30: Val Acc 0.9302
Ep 40: Val Acc 0.9302
Ep 50: Val Acc 0.9302


epoch,▁▂▂▂▃▃▄▄▄▅▅▅▆▇▇▁▃▅▅▇▇███▁▁▂▂▂▂▄▅▅▆▆▇▇███
stage,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅▅▅▅▅███████████
val_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█████▇█▇▇▇▇▆▆▆▆▅▄▄▅▃▃▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_mse,██▆▆▆▇▅▅▆▆▅▄▄▃▄▄▄▄▄▃▃▂▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch,49
stage,3
val_acc,0.93023
val_loss,1.49686
val_mse,0.00073


Task 5 Done
